# Email Spam Detection - Data Analysis & Model Training

This notebook demonstrates the complete workflow for building an email spam detection system using Machine Learning and NLP.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('../data/raw/spam.csv', encoding='latin-1')

# Display first few rows
print("Dataset Shape:", df.shape)
df.head()

## 3. Data Exploration

In [ ]:
# Dataset info
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

In [ ]:
# Class distribution
plt.figure(figsize=(10, 6))
df['label'].value_counts().plot(kind='bar', color=['#10b981', '#ef4444'])
plt.title('Distribution of Spam vs Ham Messages', fontsize=16, fontweight='bold')
plt.xlabel('Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.show()

print("\nClass Distribution:")
print(df['label'].value_counts())
print("\nPercentage:")
print(df['label'].value_counts(normalize=True) * 100)

## 4. Text Analysis

In [ ]:
# Message length analysis
df['message_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

# Statistics by class
print("Message Length Statistics:")
print(df.groupby('label')['message_length'].describe())

print("\nWord Count Statistics:")
print(df.groupby('label')['word_count'].describe())

In [ ]:
# Visualize message length distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Message length
df[df['label'] == 'ham']['message_length'].hist(bins=50, ax=axes[0], alpha=0.7, label='Ham', color='#10b981')
df[df['label'] == 'spam']['message_length'].hist(bins=50, ax=axes[0], alpha=0.7, label='Spam', color='#ef4444')
axes[0].set_title('Message Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Length')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Word count
df[df['label'] == 'ham']['word_count'].hist(bins=50, ax=axes[1], alpha=0.7, label='Ham', color='#10b981')
df[df['label'] == 'spam']['word_count'].hist(bins=50, ax=axes[1], alpha=0.7, label='Spam', color='#ef4444')
axes[1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Text Preprocessing

In [ ]:
import sys
sys.path.insert(0, '../src')

from preprocessing import TextPreprocessor

# Initialize preprocessor
preprocessor = TextPreprocessor()

# Preprocess all messages
print("Preprocessing messages...")
df['processed_text'] = df['text'].apply(preprocessor.preprocess)

print("Preprocessing completed!")

# Show examples
print("\nExample Preprocessing:")
for i in range(3):
    print(f"\nOriginal: {df['text'].iloc[i]}")
    print(f"Processed: {df['processed_text'].iloc[i]}")

## 6. Feature Extraction

In [ ]:
from feature_extraction import FeatureExtractor
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'], 
    df['label'].map({'ham': 0, 'spam': 1}),
    test_size=0.2, 
    random_state=42,
    stratify=df['label']
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Extract features
extractor = FeatureExtractor(method='tfidf', max_features=3000)
X_train_features = extractor.fit_transform(X_train)
X_test_features = extractor.transform(X_test)

print(f"\nFeature matrix shape: {X_train_features.shape}")

## 7. Model Training

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define models
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='linear', probability=True, random_state=42)
}

# Train and evaluate
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_features, y_train)
    y_pred = model.predict(X_test_features)
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred)
    }
    
    print(f"Accuracy: {results[name]['accuracy']:.4f}")
    print(f"Precision: {results[name]['precision']:.4f}")
    print(f"Recall: {results[name]['recall']:.4f}")
    print(f"F1-Score: {results[name]['f1_score']:.4f}")

## 8. Model Comparison

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df)

# Visualize comparison
results_df.plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.xlabel('Model', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Metrics')
plt.ylim(0, 1.1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

# Get best model
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_model = models[best_model_name]

print(f"Best Model: {best_model_name}")

# Predict
y_pred = best_model.predict(X_test_features)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Ham', 'Spam'],
            yticklabels=['Ham', 'Spam'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.show()

## 10. Test Predictions

In [ ]:
# Test with custom messages
test_messages = [
    "Congratulations! You've won $1000. Call now to claim your prize!",
    "Hey, can we meet for coffee tomorrow at 3 PM?",
    "URGENT: Your account will be suspended. Click here immediately!",
    "Thanks for the meeting today. I'll send the report by Friday."
]

print("Testing Custom Messages:\n")
for msg in test_messages:
    # Preprocess
    processed = preprocessor.preprocess(msg)
    # Vectorize
    features = extractor.transform([processed])
    # Predict
    prediction = best_model.predict(features)[0]
    probability = best_model.predict_proba(features)[0]
    
    print(f"Message: {msg[:60]}...")
    print(f"Prediction: {'SPAM' if prediction == 1 else 'HAM'}")
    print(f"Confidence: {probability[prediction]*100:.2f}%")
    print("-" * 80)
    print()

## 11. Save Model

In [ ]:
import joblib
import os

# Create models directory
os.makedirs('../models', exist_ok=True)

# Save best model
joblib.dump(best_model, '../models/best_model.pkl')
print(f"Model saved: ../models/best_model.pkl")

# Save vectorizer
extractor.save_vectorizer('../models/vectorizer.pkl')
print(f"Vectorizer saved: ../models/vectorizer.pkl")

# Save metadata
metadata = {
    'model_name': best_model_name,
    'accuracy': results[best_model_name]['accuracy'],
    'precision': results[best_model_name]['precision'],
    'recall': results[best_model_name]['recall'],
    'f1_score': results[best_model_name]['f1_score']
}
joblib.dump(metadata, '../models/model_metadata.pkl')
print(f"Metadata saved: ../models/model_metadata.pkl")

print("\nAll files saved successfully!")

## Conclusion

This notebook demonstrated the complete workflow for building an email spam detection system:

1. **Data Loading & Exploration**: Analyzed the dataset structure and class distribution
2. **Text Preprocessing**: Cleaned and normalized text data using NLP techniques
3. **Feature Extraction**: Converted text to numerical features using TF-IDF
4. **Model Training**: Trained multiple ML models and compared performance
5. **Evaluation**: Assessed models using accuracy, precision, recall, and F1-score
6. **Deployment**: Saved the best model for production use

The trained model can now be used in the Flask web application for real-time spam detection!